Provided an optimized helper module and minimal notebook edits to improve performance, safety and memory use. Key changes:

Replace unsafe eval with ast.literal_eval after cleaning np.float64 wrappers.
Fast NumPy implementation of get_first_one_indices.
LRU-cached kernel generator to avoid recomputing identical kernels.
Controlled parallel executor sizing and optional streaming result write to CSV to avoid building huge in-memory lists.
Small convenience wrappers.

In [ ]:
# import ast
# import re
# from functools import lru_cache
# import numpy as np
# from concurrent.futures import ThreadPoolExecutor, as_completed
# import multiprocessing
# import pandas as pd
# from typing import Any, Dict, Iterable, Callable, List, Tuple, Optional

# # import generate_kernel from the project (same import used in notebook)
# from frame_overlap import generate_kernel

# # --- Safe param parsing (avoids eval) ---
# _np_float_pattern = re.compile(r"np\.float64\(([^)]*)\)")

# def _clean_param_str(s: str) -> str:
#     if not isinstance(s, str):
#         return s
#     # remove common numpy wrapper np.float64(x) -> x
#     s2 = _np_float_pattern.sub(r"\1", s)
#     # strip possible trailing bytes/unwanted tokens
#     return s2

# def parse_param_str(param_str: Any) -> Dict[str, Any]:
#     """
#     Safely parse parameter string into a Python dict.
#     Returns empty dict on failure.
#     """
#     if isinstance(param_str, dict):
#         return param_str
#     if param_str is None:
#         return {}
#     try:
#         cleaned = _clean_param_str(param_str)
#         return ast.literal_eval(cleaned)
#     except Exception:
#         # fallback: try to exec in restricted namespace (last resort)
#         try:
#             ns = {}
#             exec("result = " + param_str, {"__builtins__": {}}, ns)
#             return ns.get("result", {}) or {}
#         except Exception:
#             return {}

# def extract_param(param_str: Any, key: str, default=float("nan")) -> float:
#     d = parse_param_str(param_str)
#     try:
#         v = d.get(key, default)
#         return float(v) if v is not None else default
#     except Exception:
#         return default

# # --- Fast first-one index detection ---
# def get_first_one_indices(mask_array: Iterable) -> List[int]:
#     """
#     Return indices where a 1 starts (first element of each contiguous run of ones).
#     Works with any iterable (list, numpy array, etc.).
#     """
#     arr = np.asarray(mask_array)
#     # consider thresholding in case array has floats close to 1
#     bool_arr = (arr == 1) if arr.dtype != bool else arr.astype(bool)
#     if bool_arr.size == 0:
#         return []
#     prev = np.concatenate(([False], bool_arr[:-1]))
#     starts = np.nonzero(bool_arr & ~prev)[0]
#     return starts.tolist()

# # --- Cached kernel generator (avoid recomputing identical kernels) ---
# # cache key includes method string if used by generate_kernel
# @lru_cache(maxsize=256)
# def cached_generate_kernel(n_pulses: int, pulse_duration: int, window_size: int, method: str = "simple") -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray]]:
#     """
#     Wrapper around project generate_kernel with caching.
#     Returns (t_kernel, kernel, slim_kernel) if slim_kernel produced; otherwise (t_kernel, kernel, None).
#     """
#     # generate_kernel in the project may accept 'method' kw; adapt as needed
#     res = generate_kernel(n_pulses=n_pulses, pulse_duration=pulse_duration, window_size=window_size, method=method)
#     # support both 2- and 3-tuple returns
#     if isinstance(res, tuple) and len(res) >= 2:
#         t_kernel, kernel = res[0], res[1]
#         slim = res[2] if len(res) > 2 else None
#         return np.asarray(t_kernel), np.asarray(kernel), (np.asarray(slim) if slim is not None else None)
#     # fallback
#     return np.asarray(res), np.asarray(res), None

# # --- Parallel runner with controlled resources and optional streaming CSV write ---
# def run_in_parallel(func: Callable, args_iterable: Iterable[Tuple], *,
#                     max_workers: Optional[int] = None,
#                     use_processes: bool = False,
#                     stream_csv: Optional[str] = None,
#                     csv_chunksize: int = 32) -> List[Dict]:
#     """
#     Run func(*args) in parallel over args_iterable.
#     - max_workers: if None uses cpu_count()-1 (at least 1).
#     - use_processes: if True uses multiprocessing.ProcessPoolExecutor (picklability required).
#     - stream_csv: if given path, results are appended to CSV as they arrive to avoid big RAM usage.
#     Returns list of results (if streaming to CSV, still returns list but may be large).
#     """
#     if max_workers is None:
#         cpus = multiprocessing.cpu_count()
#         max_workers = max(1, cpus - 1)

#     executor_cls = ThreadPoolExecutor
#     # Note: using ProcessPoolExecutor may fail for objects that are not picklable.
#     from concurrent.futures import ThreadPoolExecutor as TPE
#     from concurrent.futures import ProcessPoolExecutor as PPE
#     executor_cls = PPE if use_processes else TPE

#     results: List[Dict] = []
#     # streaming setup
#     csv_file = None
#     header_written = False
#     if stream_csv:
#         csv_file = open(stream_csv, "w", newline="")
#         csv_file.close()  # will append below

#     with executor_cls(max_workers=max_workers) as executor:
#         futures = {executor.submit(func, *args): args for args in args_iterable}
#         for fut in as_completed(futures):
#             try:
#                 r = fut.result()
#             except Exception as e:
#                 r = {"error": str(e)}
#             if stream_csv:
#                 # append to CSV in batches (here per result)
#                 try:
#                     df = pd.DataFrame([r])
#                     df.to_csv(stream_csv, mode="a", index=False, header=not header_written)
#                     header_written = True
#                 except Exception:
#                     # best-effort: skip writing on failure
#                     pass
#             results.append(r)
#     return results

Notebook edits (minimal replacements). Replace the unsafe eval-based extract_param, get_first_one_indices and kernel calls with optimized helpers. Add import at top of the notebook (modify the cell that imports package functions):

In [1]:
import ast
import re
from functools import lru_cache
import numpy as np
import pandas as pd
import multiprocessing
from typing import Any, Dict, Iterable, Callable, List, Tuple, Optional
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from itertools import product
import matplotlib.pyplot as plt


# import project's generate_kernel (used by cached wrapper)
from frame_overlap import generate_kernel
import nbragg

In [2]:


_np_float_pattern = re.compile(r"np\.float64\(([^)]*)\)")

def _clean_param_str(s: Any) -> Any:
    if not isinstance(s, str):
        return s
    return _np_float_pattern.sub(r"\1", s)

def parse_param_str(param_str: Any) -> Dict[str, Any]:
    """
    Safely parse a parameter-string (previously used with eval) into a dict.
    Returns {} on failure.
    """
    if isinstance(param_str, dict):
        return param_str
    if param_str is None:
        return {}
    try:
        cleaned = _clean_param_str(param_str)
        return ast.literal_eval(cleaned)
    except Exception:
        # last-resort restricted exec (very limited)
        try:
            ns: Dict[str, Any] = {}
            exec("result = " + str(param_str), {"__builtins__": {}}, ns)
            return ns.get("result", {}) or {}
        except Exception:
            return {}

def extract_param(param_str: Any, key: str, default: float = float("nan")) -> float:
    d = parse_param_str(param_str)
    try:
        v = d.get(key, default)
        return float(v) if v is not None else default
    except Exception:
        return default

def get_first_one_indices(mask_array: Iterable) -> List[int]:
    """
    Fast numpy-based detection of start indices of contiguous runs of 1s.
    """
    arr = np.asarray(mask_array)
    if arr.size == 0:
        return []
    # normalize to boolean: treat values equal to 1 as True
    if arr.dtype == bool:
        bool_arr = arr
    else:
        bool_arr = (arr == 1)
    prev = np.concatenate(([False], bool_arr[:-1]))
    starts = np.nonzero(bool_arr & ~prev)[0]
    return starts.tolist()

@lru_cache(maxsize=256)
def cached_generate_kernel(n_pulses: int, pulse_duration: int, window_size: int, method: str = "simple"):
    """
    Cached wrapper for the project's generate_kernel function.
    Returns (t_kernel, kernel, slim_kernel_or_None)
    """
    res = generate_kernel(n_pulses=n_pulses, pulse_duration=pulse_duration, window_size=window_size, method=method)
    if isinstance(res, tuple) and len(res) >= 2:
        t_kernel, kernel = res[0], res[1]
        slim = res[2] if len(res) > 2 else None
        return np.asarray(t_kernel), np.asarray(kernel), (np.asarray(slim) if slim is not None else None)
    # fallback: return res duplicated
    arr = np.asarray(res)
    return arr, arr, None

def run_in_parallel(func: Callable, args_iterable: Iterable[Tuple], *,
                    max_workers: Optional[int] = None,
                    use_processes: bool = False,
                    stream_csv: Optional[str] = None) -> List[Dict]:
    """
    Run func(*args) for each args tuple in args_iterable in parallel.
    If stream_csv is provided, results are appended to the CSV incrementally (reduces peak memory).
    Returns the list of results (may still be large, but streaming avoids building huge in-memory lists at once).
    """
    cpus = multiprocessing.cpu_count()
    if max_workers is None:
        max_workers = max(1, cpus - 1)
    executor_cls = ProcessPoolExecutor if use_processes else ThreadPoolExecutor

    results: List[Dict] = []
    header_written = False
    if stream_csv:
        # ensure file exists and is empty
        with open(stream_csv, "w") as f:
            f.write("")

    with executor_cls(max_workers=max_workers) as executor:
        futures = {executor.submit(func, *args): args for args in args_iterable}
        for fut in as_completed(futures):
            try:
                r = fut.result()
            except Exception as e:
                r = {"error": str(e)}
            # stream to CSV if requested
            if stream_csv:
                try:
                    df = pd.DataFrame([r])
                    df.to_csv(stream_csv, mode="a", index=False, header=not header_written)
                    header_written = True
                except Exception:
                    pass
            results.append(r)
    return results

In [3]:
# ...existing code...
from frame_overlap import (
    read_tof_data,
    prepare_full_frame,
    generate_kernel,
    apply_filter,
    chi2_analysis,
    plot_analysis,
    optimize_parameters
)
# # new optimized helpers
# from optimized_utils import (
#     extract_param,
#     get_first_one_indices,
#     cached_generate_kernel,
#     run_in_parallel
# )
# import multiprocessing
# ...existing code...

In [4]:
nbragg.utils.register_material("Fe_sg229_Iron-alpha_CrysExtn1.ncmat")
nbragg.utils.register_material("Cellulose_C6O5H10.ncmat")
# xs = nbragg.CrossSection.from_material("Fe_sg229_Iron-alpha.ncmat")
# xs = nbragg.CrossSection(iron="Fe_sg229_Iron-alpha.ncmat")                      # define sample

# xs0 = 0.0275 * nbragg.CrossSection(cellulose="Cellulose_C6O5H10.ncmat") + \
#      (1 - 0.0275) * nbragg.CrossSection(α="Fe_sg229_Iron-alpha_CrysExtn1.ncmat")
xs0 = 0.0275 * nbragg.CrossSection(cellulose=nbragg.materials["Cellulose_C6O5H10"]) + \
     (1 - 0.0275) * nbragg.CrossSection(α=nbragg.materials["Fe_sg229_Iron-alpha_CrysExtn1"])

# xs = nbragg.CrossSection(alpha=nbragg.materials["Fe_sg229_Iron-alpha.ncmat"])*0.98 + nbragg.CrossSection(water=nbragg.materials["LiquidWaterH2O_T300.0K.ncmat"])*0.02


xs0.materials['α']["ext_method"] = "Sabine_uncorr"
xs0.materials['α']["ext_tilt"] = "tri"
xs0 = nbragg.CrossSection(xs0.materials)

xs1 = 0.0275 * nbragg.CrossSection(cellulose=nbragg.materials["Cellulose_C6O5H10"]) + \
     (1 - 0.0275) * nbragg.CrossSection(α=nbragg.materials["Fe_sg229_Iron-alpha_CrysExtn1"])
xs1.materials['α']["ext_method"] = "Sabine_uncorr"
xs1.materials['α']["ext_tilt"] = "rect"
xs1 = nbragg.CrossSection(xs1.materials)

xs2 = 0.0275 * nbragg.CrossSection(cellulose=nbragg.materials["Cellulose_C6O5H10"]) + \
     (1 - 0.0275) * nbragg.CrossSection(α=nbragg.materials["Fe_sg229_Iron-alpha_CrysExtn1"])
xs2.materials['α']["ext_method"] = "Sabine_corr"
xs2 = nbragg.CrossSection(xs2.materials)

xs3 = 0.0275 * nbragg.CrossSection(cellulose=nbragg.materials["Cellulose_C6O5H10"]) + \
     (1 - 0.0275) * nbragg.CrossSection(α=nbragg.materials["Fe_sg229_Iron-alpha_CrysExtn1"])
xs3.materials['α']["ext_method"] = "Sabine_corr"
xs3 = nbragg.CrossSection(xs3.materials)

xs4 = 0.0275 * nbragg.CrossSection(cellulose=nbragg.materials["Cellulose_C6O5H10"]) + \
     (1 - 0.0275) * nbragg.CrossSection(α=nbragg.materials["Fe_sg229_Iron-alpha_CrysExtn1"])
xs4.materials['α']["ext_method"] = "BC_mix"
xs4 = nbragg.CrossSection(xs4.materials)

xs5 = 0.0275 * nbragg.CrossSection(cellulose=nbragg.materials["Cellulose_C6O5H10"]) + \
     (1 - 0.0275) * nbragg.CrossSection(α=nbragg.materials["Fe_sg229_Iron-alpha_CrysExtn1"])
xs5.materials['α']["ext_method"] = "CR"
xs5 = nbragg.CrossSection(xs5.materials)

# model = nbragg.TransmissionModel(xs, vary_extinction=True,vary_background=True,vary_weights=True,vary_response=True)
# model = nbragg.TransmissionModel(xs, vary_extinction=True,vary_weights=True,vary_response=True)

groups0 = {
    "basic": ["norm"],
    "thickness": ["thickness"],
    "extinction": ["ext_Gg2", "ext_L2"],
    "bg": ["background"],
    "weights": ["weights"],
    "response": ["response"]    
}

groups1 = {
    "basic": ["norm"],
    "bg": ["background"],
    "extinction": ["ext_Gg2", "ext_L2"],
    "thickness": ["thickness"],
    "weights": ["weights"],
    "response": ["response"]    
}

groups2 = {
    "basic": ["norm","thickness"],
    "bg": ["background"],
    "Advanced": ["ext_Gg2", "ext_L2","weights","response"]   
}

groups3 = {
    "basic": ["norm","thickness","wlmin=3","wlmax=6"],
    "bg": ["background","wlmin=3","wlmax=5"],
    "Advanced": ["ext_Gg2", "ext_L2","weights","response","wlmin=2","wlmax=4.2"]   
}

groups4 = {
    "basic": ["norm","thickness","wlmin=1.5","wlmax=4.5"],
    "bg": ["background","wlmin=1.5","wlmax=4.5"],
    "weights": ["weights","wlmin=1.5","wlmax=4.5"],
    "advanced": ["ext_Gg2", "ext_L2","wlmin=1.5","wlmax=4.5"],
    "response": ["response","wlmin=1.5","wlmax=4.5"]  
}

groups5 = {
    "norm": ["norm"],
    "bg": ["background"],
    "thickness": ["thickness"],
    "weights": ["weights"]
}

groups6 = {
    "thickness": ["thickness"],
    "weights": ["weights"]
}

xs_dict = {
    'xs0': xs0,
    'xs1': xs1,
    'xs2': xs2, 
    'xs3': xs3,
    'xs4': xs4,
    'xs5': xs5
}
groups_dict = {
    'None': None,
    'groups0': groups0,
    'groups1': groups1,
    'groups2': groups2,
    'groups3': groups3,
    'groups4': groups4,
    'groups5': groups5,
    'groups6': groups6
}

t_signal, signal, errors, stacks = read_tof_data('iron_powder.csv')
t_signal_ob, signal_ob, errors_ob, stacks_ob = read_tof_data('openbeam.csv')

stacks1, signal1, errors1 = prepare_full_frame(t_signal, signal, errors, stacks, max_stack=5000)
stacks_ob1, signal_ob1, errors_ob1 = prepare_full_frame(t_signal_ob, signal_ob, errors_ob, stacks_ob, max_stack=5000)

In [5]:
def write_data_df(stack,signal,error):
    df=pd.DataFrame()
    df['stack']=stack
    signal[signal<0]=np.nan
    df['signal']=np.round(signal,3)
    # error[error==np.isnan(error)]=np.nan
    df['error']=np.round(error,3)
    # df.to_csv(filepath,sep=',')
    return df

def construct_data(signal,signal_ob,n_pulses=5,pulse_duration=100,bin_width=10,filter_type='simple'):
    saraf_irad_time = 24 # time in hours
    frequency = 20 # frequency in herz
    n_pulses = n_pulses # number of pulses
    pulse_duration = pulse_duration # pulse duration in usec
    window_size=int(len(signal)*bin_width)  
    # print(window_size)  
    scaling_factor = (1e6/5e6) * (saraf_irad_time/0.5) * (frequency*n_pulses*pulse_duration*1e-6)
    scaled_signal = signal * scaling_factor
    scaled_ob = signal_ob * scaling_factor
    # if filter_type == 'simple':
    t_kernel, kernel, slim_kernel = generate_kernel(n_pulses=n_pulses, pulse_duration=pulse_duration, window_size=window_size, method=filter_type)
    # elif filter_type == 'complex':
        # t_kernel, kernel = generate_kernel(n_pulses=n_pulses, pulse_duration=pulse_duration, window_size=window_size)
    
    # Apply Wiener deconvolution to the signal
    observed_poisson, reconstructed = apply_filter(
        scaled_signal,
        kernel,
        slim_kernel,
        # stats_fraction=1,
        noise_power=0.005
    )
    # Apply Wiener deconvolution to the signal
    observed_poisson_ob, reconstructed_ob = apply_filter(
        scaled_ob,
        kernel,
        slim_kernel,
        # stats_fraction=1,
        noise_power=0.005
    )
    # plt.plot(kernel)
    return reconstructed, reconstructed_ob, kernel


def run_nbragg(n_start=50, pulse_width=30, xs=xs0, filter_type='single', groups=groups0, modified=True):
    if modified:
        reconstructed, reconstructed_ob, kernel = construct_data(signal1, signal_ob1, n_start, pulse_width, filter_type=filter_type)
        iron_powder1 = write_data_df(stacks1, reconstructed, np.sqrt(reconstructed))
        openbeam1 = write_data_df(stacks_ob1, reconstructed_ob, np.sqrt(reconstructed_ob))
        data = nbragg.Data.from_counts(iron_powder1, openbeam1)
    else:
        data = nbragg.Data.from_counts("iron_powder.csv", "openbeam.csv")
    
    data.table = data.table.dropna()
    # model = nbragg.TransmissionModel(xs,response="square_jorgensen",vary_background=True,vary_response=True)

    if groups is None:
        model = nbragg.TransmissionModel(
            xs,
            response="square_jorgensen",
            vary_extinction=True,
            vary_background=True,
            vary_weights=True,
            vary_response=True
        )
        result = model.fit(data)
    elif groups == groups5:
        model = nbragg.TransmissionModel(
            xs,
            response="square_jorgensen",
            vary_background=True,
            vary_weights=True,
            vary_response=True
        )
        result = model.fit(data, method="rietveld", stages=groups, progress_bar=False)
    elif groups == groups6:
        model = nbragg.TransmissionModel(
            xs,
            response="square_jorgensen",
            vary_weights=True
        )
        result = model.fit(data, method="rietveld", stages=groups, progress_bar=False)
    else:
        model = nbragg.TransmissionModel(
            xs,
            response="square_jorgensen",
            vary_background=True,
            vary_weights=True,
            vary_extinction=True,
            vary_response=True
        )
        result = model.fit(
            data,
            method="rietveld",
            stages=groups,
            progress_bar=False
        )
    # minimizer_result = result.minimizer
    # Force calculation of uncertainties (covariance matrix)
    # ci = lmfit.conf_interval(result.minimizer, result)
    # result.calc_covar()

    # Extract initial and final parameters
    # n_indexes = {get_first_one_indices(kernel)}
    initial_params = {k: v.value for k, v in result.init_params.items()}
    final_params = {k: v.value for k, v in result.params.items()}
    uncertainties = {k: v.stderr for k, v in result.params.items()}
    # plt.plot(kernel)
    return {
        'first_n_indices': get_first_one_indices(kernel),
        # 'last_n_indices': get_last_one_indices(kernel),
        'pulse_width': pulse_width,
        'initial_params': initial_params,
        'final_params': final_params,
        'uncertainties': uncertainties,
        'redchi': result.redchi,
        'success': result.success,
    }
    # return result



def run_with_context(n_start, pulse_width, xs_name, groups_name, modified,filter_type):
    # pref_to_find1 = "groups"
    # group_names = example_function(pref_to_find1)
    # # group_names = find_variables_by_prefix(pref_to_find1, locals())
    # pref_to_find2 = "xs"
    # xs_names = find_variables_by_prefix(pref_to_find2, locals())
    xs = xs_dict[xs_name]  # Retrieve the object from xs_dict using the name
    groups = groups_dict[groups_name]
    
    try:
        result = run_nbragg(n_start=n_start, pulse_width=pulse_width, xs=xs, groups=groups, modified=modified,filter_type=filter_type)
        # uncertainties_dict = {name: param.stderr for name, param in result.params.items() if param.stderr is not None}
        # print('result values',result.params.valuesdict())
        # print('uncertainty values',uncertainties_dict)
        return {
            # 'n_start': n_start,
            # 'pulse_width': pulse_width,
            'xs': xs_name,     # You may need a way to stringify your xs/group objects
            'groups': groups_name,
            'filter_type': filter_type,
            **result
            # 'params': result.params.valuesdict(),
            # 'uncertainties': uncertainties_dict,
            # 'redchi': result.redchi,
            # 'success': result.success
        }
    except Exception as e:
        return {
            # 'n_start': n_start,
            # 'pulse_width': pulse_width,
            'xs': xs_name,
            'groups': groups_name,
            'filter_type': filter_type,
            'error': str(e)
        }
    
# Add a helper cell: analyze_single_case
# ...existing code...

import matplotlib.pyplot as plt

def analyze_single_case(n_start, pulse_width, xs_name='xs0', groups_name='groups4', modified=True, filter_type='simple', window_size=None, show=True):
    """
    Construct data for a single case, run the filter, run nbragg fit and plot:
      - observed poisson (deconvolution input/output),
      - reconstructed (deconvolution output),
      - kernel and slim kernel (if available),
      - a simple comparison: pulse_width vs fitted square-response width (if present in final_params).
    Returns the nbragg fit result dict from run_with_context (or error dict).
    """
    # prepare kernel (use cached generator)
    if window_size is None:
        # derive window_size from prepared signal if available in notebook (fallback to 50000)
        try:
            window_size = len(signal1) * 10
        except Exception:
            window_size = 50000

    t_kernel, kernel, slim_kernel = cached_generate_kernel(n_pulses=n_start, pulse_duration=pulse_width, window_size=window_size, method=filter_type)

    # construct scaled & filtered data
    reconstructed, reconstructed_ob, _kernel = construct_data(signal, signal_ob, n_pulses=n_start, pulse_duration=pulse_width, bin_width=10, filter_type=filter_type)

    # Build observed poisson via apply_filter directly (so we can show observed)
    saraf_irad_time = 24
    frequency = 20
    scaling_factor = (1e6/5e6) * (saraf_irad_time/0.5) * (frequency * n_start * pulse_width * 1e-6)
    try:
        scaled_signal = signal1 * scaling_factor
        scaled_ob = signal_ob1 * scaling_factor
    except Exception:
        scaled_signal = signal * scaling_factor
        scaled_ob = signal_ob * scaling_factor

    observed_poisson, reconstructed_local = apply_filter(scaled_signal, kernel, slim_kernel, noise_power=0.005)

    # run nbragg fit for this single case (use run_with_context wrapper)
    fit_result = run_with_context(n_start=n_start, pulse_width=pulse_width, xs_name=xs_name, groups_name=groups_name, modified=modified, filter_type=filter_type)

    # extract fitted response width if present (common names tried)
    final_params = fit_result.get('final_params', {})
    resp_width = extract_param(final_params, 'response') if final_params else float('nan')
    # some models use 'response_width' or 'response_fwhm' etc.
    if np.isnan(resp_width):
        for alt in ('response_width', 'response_fwhm', 'resp_width', 'width'):
            v = extract_param(final_params, alt)
            if not np.isnan(v):
                resp_width = v
                break

    # plotting
    if show:
        fig, axs = plt.subplots(2, 1, figsize=(10, 8), constrained_layout=True)

        # top: signals & reconstruction
        ax = axs[0]
        ax.plot(np.arange(len(scaled_signal)), scaled_signal, label='scaled_signal', alpha=0.6)
        ax.plot(np.arange(len(reconstructed_local)), reconstructed_local, label='reconstructed', alpha=0.8)
        ax.plot(np.arange(len(scaled_ob)), scaled_ob, label='scaled_openbeam', alpha=0.6)
        ax.set_title(f'scaled signals and reconstruction (n_start={n_start}, pulse_width={pulse_width})')
        ax.set_xlabel('time index')
        ax.set_ylabel('counts (scaled)')
        ax.legend()

        # overlay kernel on twin axis (scaled)
        axk = ax.twinx()
        kscale = max(1.0, np.nanmax(scaled_signal)) if np.nanmax(scaled_signal) != 0 else 1.0
        axk.plot(np.arange(min(len(kernel), len(scaled_signal))), kernel[:min(len(kernel), len(scaled_signal))] * kscale, color='gray', alpha=0.5, label='kernel (scaled)')
        if slim_kernel is not None:
            axk.plot(np.arange(min(len(slim_kernel), len(scaled_signal))), slim_kernel[:min(len(slim_kernel), len(scaled_signal))] * kscale, color='orange', alpha=0.6, label='slim_kernel (scaled)')
        axk.set_ylabel('kernel * scale')
        axk.legend(loc='upper right')

        # bottom: fit response width vs pulse width
        ax2 = axs[1]
        ax2.bar(['pulse_width', 'fitted_response'], [pulse_width, resp_width], color=['tab:blue', 'tab:orange'])
        ax2.set_ylabel('width (µs)')
        ax2.set_title('Pulse width vs fitted square-response width')
        plt.show()

    return fit_result

# ...existing code...

In [ ]:
# Replace the large ThreadPoolExecutor parameter-scan cell (previously using max_workers=35)
# ...existing code...
# (This cell replaces the prior param-combination + ThreadPoolExecutor block that wrote
#  ../results/fit_results_24hrs_saraf_simple_groups4_1000.csv)
xs_list = ['xs0']
groups_list = ['groups4']

n_start_range = range(2, 25, 1)
pulse_width_range = range(100, 1000, 100)
modified = [True]
filter_type = ['simple']

param_combinations = list(product(n_start_range, pulse_width_range, xs_list, groups_list, modified, filter_type))

# population_range = range(len(param_combinations))  # Represents numbers from 1 to len(param_combinations)
# num_elements_to_choose = 10

# # Randomly select the indices
# random_elements = np.random.choice(population_range, num_elements_to_choose, replace=False)

# # Create the mask
# mask = np.zeros(len(param_combinations), dtype=bool)
# mask[random_elements] = True

# # Apply the mask to the param_combinations list
# # param_combinations1 = np.array(param_combinations)[mask]
# param_combinations_updated = [param_combinations[i] for i in range(len(param_combinations)) if mask[i]]

# convert to args iterable (each element is a tuple of args for run_with_context)
args_iterable = [
    (n_start, pulse_width, xs_name, groups_name, modified_flag, ftype)
    for (n_start, pulse_width, xs_name, groups_name, modified_flag, ftype) in param_combinations
]

# wrapper that adapts run_with_context (already defined in notebook) for parallel execution
def run_case(n_start, pulse_width, xs_name, groups_name, modified_flag, ftype):
    # run_with_context returns a dict or error dict
    return run_with_context(n_start=n_start, pulse_width=pulse_width, xs_name=xs_name,
                            groups_name=groups_name, modified=modified_flag, filter_type=ftype)

out_csv = '../results/fit_results_24hrs_saraf_simple_groups4_%d_271025.csv'%len(param_combinations)
max_workers = min(35, max(1, multiprocessing.cpu_count() - 1))

# stream results to CSV to avoid huge in-memory lists
results = run_in_parallel(run_case, args_iterable, max_workers=max_workers, use_processes=False, stream_csv=out_csv)
# results list is still returned; if size is large consider deleting it
del results
# ...existing code...

/home/saraf/miniforge3/envs/analysis-env/lib/python3.13/site-packages/nbragg/models.py:1444: UserWarning: @CRYSEXTN section is not defined for the cellulose phase
  warnings.warn(f"@CRYSEXTN section is not defined for the {material} phase")
/home/saraf/miniforge3/envs/analysis-env/lib/python3.13/site-packages/nbragg/models.py:1444: UserWarning: @CRYSEXTN section is not defined for the α phase
  warnings.warn(f"@CRYSEXTN section is not defined for the {material} phase")


In [ ]:
# Replace the large ThreadPoolExecutor parameter-scan cell (previously using max_workers=35)
# ...existing code...
# (This cell replaces the prior param-combination + ThreadPoolExecutor block that wrote
#  ../results/fit_results_24hrs_saraf_simple_groups4_1000.csv)
xs_list = ['xs0']
groups_list = ['groups4']

n_start_range = range(2, 25, 1)
pulse_width_range = range(100, 1000, 100)
modified = [True]
filter_type = ['poisson']

param_combinations = list(product(n_start_range, pulse_width_range, xs_list, groups_list, modified, filter_type))

# population_range = range(len(param_combinations))  # Represents numbers from 1 to len(param_combinations)
# num_elements_to_choose = 10

# # Randomly select the indices
# random_elements = np.random.choice(population_range, num_elements_to_choose, replace=False)

# # Create the mask
# mask = np.zeros(len(param_combinations), dtype=bool)
# mask[random_elements] = True

# # Apply the mask to the param_combinations list
# # param_combinations1 = np.array(param_combinations)[mask]
# param_combinations_updated = [param_combinations[i] for i in range(len(param_combinations)) if mask[i]]

# convert to args iterable (each element is a tuple of args for run_with_context)
args_iterable = [
    (n_start, pulse_width, xs_name, groups_name, modified_flag, ftype)
    for (n_start, pulse_width, xs_name, groups_name, modified_flag, ftype) in param_combinations
]

# wrapper that adapts run_with_context (already defined in notebook) for parallel execution
def run_case(n_start, pulse_width, xs_name, groups_name, modified_flag, ftype):
    # run_with_context returns a dict or error dict
    return run_with_context(n_start=n_start, pulse_width=pulse_width, xs_name=xs_name,
                            groups_name=groups_name, modified=modified_flag, filter_type=ftype)

out_csv = '../results/fit_results_24hrs_saraf_poisson_groups4_%d_271025.csv'%len(param_combinations)
max_workers = min(35, max(1, multiprocessing.cpu_count() - 1))

# stream results to CSV to avoid huge in-memory lists
results = run_in_parallel(run_case, args_iterable, max_workers=max_workers, use_processes=False, stream_csv=out_csv)
# results list is still returned; if size is large consider deleting it
del results
# ...existing code...

Notes and recommended quick wins

Replace every use of eval(param_str) with extract_param or parse_param_str to avoid code injection and speed repeated parsing.
Use cached_generate_kernel whenever the same kernel parameters may repeat (common in parameter sweeps).
When producing large result sets, use run_in_parallel(..., stream_csv="path/to/out.csv") to avoid keeping all results in memory.
Limit parallelism to CPU count minus one. If nbragg or lmfit are not thread-safe, test ProcessPoolExecutor carefully — many complex objects are not picklable.
Replace repeated small DataFrame.to_csv calls by buffering a few results and writing in chunks for higher throughput (run_in_parallel can be extended to buffer).
If you want, I can:

Apply these notebook edits for you (update cells).
Convert a large parameter-scan cell to use run_in_parallel with streaming CSV output.
Which change should I apply next?